In this section, we will be working with the UNSW-NB15 dataset, a widely used benchmark dataset for network intrusion and anomaly detection. Here rather than using merely the volume of traffic we will be trying to identify attacks with a high degree of accuracy and speed using other features of the requests and their origins.

The goal of this assignment is to explore how machine learning models can be used to detect attacks with both high accuracy and low latency. In particular, we will focus on gradient boosting methods, which are well-suited for tabular data and can capture complex nonlinear relationships between features. You will implement and evaluate a LightGBM-based model, examining how it performs relative to alternative approaches in terms of predictive power and computational efficiency.


This exercise is inspired by [this](https://ieeexplore.ieee.org/document/9315049) paper, which tests LightGBM's performance relative to its alternatives.

## Setup and Imports

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 808
TEST_SIZE = 0.2

_DATA = Path.cwd() / "DataFiles"
DATA_PATH = str(_DATA / "UNSW-NB15_1.csv")
FEATURE_PATH = str(_DATA / "NUSW-NB15_features.csv")


## Load Data and Assign Canonical Column Names

In [2]:
feature_df = pd.read_csv(FEATURE_PATH, encoding='cp1252')
feature_names = feature_df["Name"].astype(str).str.strip().str.replace(" ", "", regex=False).tolist()

if len(feature_names) != 49:
    raise ValueError(f"Expected 49 feature names, got {len(feature_names)}")

df = pd.read_csv(DATA_PATH, header=None)
if df.shape[1] != len(feature_names):
    raise ValueError(f"Data has {df.shape[1]} columns, but schema has {len(feature_names)}")

df.columns = feature_names
print(f"Loaded shape: {df.shape}")
print(f"Columns assigned: {len(df.columns)}")
df.head(2)

Loaded shape: (700001, 49)
Columns assigned: 49


/var/folders/69/gr9hpxt508l7wq2jn8rnx84w0000gn/T/ipykernel_25359/1163756759.py:7: DtypeWarning: Columns (0: 1, 1: 3, 2: 47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH, header=None)


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0


## Preprocessing and Feature Construction

In [3]:
df = df.copy()

# Categorical columns per UNSW-NB15 schema (nominal fields).
categorical_cols = [c for c in ["proto", "state", "service"] if c in df.columns]

# Normalize targets first.
df["attack_cat"] = (
    df["attack_cat"]
    .fillna("Normal")
    .astype(str)
    .str.strip()
    .replace({"": "Normal", "-": "Normal"})
)
df["Label"] = pd.to_numeric(df["Label"], errors="coerce").fillna(0).astype(int)

# Drop leakage / identifiers from features.
drop_for_modeling = ["srcip", "dstip", "Stime", "Ltime"]
drop_for_modeling = [c for c in drop_for_modeling if c in df.columns]

X_raw = df.drop(columns=drop_for_modeling + ["Label", "attack_cat"])
y_binary = df["Label"]
y_multi_raw = df["attack_cat"]

# Coerce numeric columns (ports sometimes load as strings).
for c in X_raw.columns:
    if c not in categorical_cols:
        X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")
X_raw = X_raw.fillna(0)

# Log-transform heavy-tailed nonnegative numeric features.
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols:
    s = X_raw[c]
    if (s.min(skipna=True) or 0) >= 0:
        X_raw[c] = np.log1p(np.maximum(s, 0))

X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=False)
print(f"Model feature matrix shape after one-hot encoding: {X.shape}")


Model feature matrix shape after one-hot encoding: (700001, 204)


In [4]:
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X,
    y_binary,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_binary,
)

print("Binary split shapes:")
print("X_train:", X_train_bin.shape, "X_test:", X_test_bin.shape)
print("y_train mean:", y_train_bin.mean(), "y_test mean:", y_test_bin.mean())
print("Binary train distribution:")
print(y_train_bin.value_counts(normalize=True).sort_index())

Binary split shapes:
X_train: (560000, 204) X_test: (140001, 204)
y_train mean: 0.031735714285714284 y_test mean: 0.031735487603659976
Binary train distribution:
Label
0    0.968264
1    0.031736
Name: proportion, dtype: float64


XGBoost Baseline Model (you shouldn't need to edit the cell below)

In [5]:
neg_count = int((y_train_bin == 0).sum())
pos_count = int((y_train_bin == 1).sum())
scale_pos_weight = (neg_count / pos_count) if pos_count > 0 else 1.0

xgb_model = XGBClassifier(
    random_state=RANDOM_STATE,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
)

t0 = time.perf_counter()
xgb_model.fit(X_train_bin, y_train_bin)
xgb_train_seconds = time.perf_counter() - t0

t1 = time.perf_counter()
xgb_preds = xgb_model.predict(X_test_bin)
xgb_infer_seconds = time.perf_counter() - t1

xgb_infer_full_est_seconds = xgb_infer_seconds * (len(X) / len(X_test_bin))
xgb_total_load_test_split = xgb_train_seconds + xgb_infer_seconds
xgb_total_load_full_est = xgb_train_seconds + xgb_infer_full_est_seconds
xgb_acc = accuracy_score(y_test_bin, xgb_preds)

print(
    "XGBoost (scale_pos_weight): "
    f"train={xgb_train_seconds:.4f}s, infer={xgb_infer_seconds:.4f}s, "
    f"total_load={xgb_total_load_test_split:.4f}s, acc={xgb_acc:.4f}"
)

xgb_result = {
    "Model": "XGBoost (scale_pos_weight)",
    "Train Time (s)": xgb_train_seconds,
    "Inference Time (s)": xgb_infer_seconds,
    "Total Load (s)": xgb_total_load_test_split,
    "Estimated Full Inference Load (s)": xgb_infer_full_est_seconds,
    "Estimated Full Total Load (s)": xgb_total_load_full_est,
    "Binary Accuracy": xgb_acc,
    "Imbalance Handling": f"scale_pos_weight={scale_pos_weight:.4f}",
}

XGBoost (scale_pos_weight): train=4.9042s, infer=0.0519s, total_load=4.9561s, acc=0.9960


## 6) Binary LightGBM Models: Vanilla vs Bundled vs GOSS

In [6]:
def run_binary_experiment(model_name, model_kwargs):
    binary_imbalance_kwargs = {"is_unbalance": True}
    kw = dict(
        random_state=RANDOM_STATE,
        n_estimators=200,
        learning_rate=0.1,
        colsample_bytree=1.0,
        **binary_imbalance_kwargs,
        **model_kwargs,
    )
    kw.setdefault("max_depth", 6)
    kw.setdefault("subsample", 1.0)
    model = LGBMClassifier(**kw)

    t0 = time.perf_counter()
    model.fit(X_train_bin, y_train_bin)
    train_seconds = time.perf_counter() - t0

    t1 = time.perf_counter()
    preds = model.predict(X_test_bin)
    infer_seconds = time.perf_counter() - t1

    infer_full_est_seconds = infer_seconds * (len(X) / len(X_test_bin))
    total_load_test_split = train_seconds + infer_seconds
    total_load_full_est = train_seconds + infer_full_est_seconds

    acc = accuracy_score(y_test_bin, preds)
    print(
        f"{model_name}: train={train_seconds:.4f}s, infer={infer_seconds:.4f}s, "
        f"total_load={total_load_test_split:.4f}s, acc={acc:.4f}"
    )

    return {
        "Model": model_name,
        "Train Time (s)": train_seconds,
        "Inference Time (s)": infer_seconds,
        "Total Load (s)": total_load_test_split,
        "Estimated Full Inference Load (s)": infer_full_est_seconds,
        "Estimated Full Total Load (s)": total_load_full_est,
        "Binary Accuracy": acc,
        "Imbalance Handling": "is_unbalance=True",
    }


binary_results = [xgb_result]

binary_results.append(
    run_binary_experiment("Vanilla LGBM", {"max_depth": 6})
)
binary_results.append(
    run_binary_experiment(
        "Deeper trees (num_leaves=127)",
        {"num_leaves": 127, "max_depth": 12},
    )
)
binary_results.append(
    run_binary_experiment(
        "GOSS boosting",
        {"boosting_type": "goss"},
    )
)
binary_results.append(
    run_binary_experiment(
        "Bagging 50% row subsample",
        {
            "subsample": 0.5,
            "subsample_freq": 1,
        },
    )
)


[LightGBM] [Info] Number of positive: 17772, number of negative: 542228
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028713 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5886
[LightGBM] [Info] Number of data points in the train set: 560000, number of used features: 80
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.031736 -> initscore=-3.418062
[LightGBM] [Info] Start training from score -3.418062
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Vanilla LGBM: train=2.2396s, infer=0.1069s, total_load=2.3465s, acc=0.9961


[LightGBM] [Info] Number of positive: 17772, number of negative: 542228
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019079 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5886
[LightGBM] [Info] Number of data points in the train set: 560000, number of used features: 80
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.031736 -> initscore=-3.418062
[LightGBM] [Info] Start training from score -3.418062
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
GOSS boosting: train=2.5799s, infer=0.0964s, total_load=2.6763s, acc=0.9961


[LightGBM] [Info] Number of positive: 17772, number of negative: 542228
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019338 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5886
[LightGBM] [Info] Number of data points in the train set: 560000, number of used features: 80
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.031736 -> initscore=-3.418062
[LightGBM] [Info] Start training from score -3.418062
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Bagging 50% row subsample: train=2.6895s, infer=0.1011s, total_load=2.7905s, acc=0.9957


# Comparison Table:

In [7]:
comparison_df = pd.DataFrame(binary_results)
comparison_df = comparison_df.sort_values(by="Binary Accuracy", ascending=False).reset_index(drop=True)
comparison_df = comparison_df[
    [
        "Model",
        "Train Time (s)",
        "Inference Time (s)",
        "Total Load (s)",
        "Estimated Full Inference Load (s)",
        "Estimated Full Total Load (s)",
        "Binary Accuracy",
        "Imbalance Handling",
    ]
]
comparison_df

,Model,Train Time (s),Inference Time (s),Total Load (s),Estimated Full Inference Load (s),Estimated Full Total Load (s),Binary Accuracy,Imbalance Handling
0,Deeper trees (num_leaves=127),5.516381,0.166580,5.682960,0.832893,6.349273,0.997664,is_unbalance=True
1,GOSS boosting,2.579910,0.096405,2.676315,0.482024,3.061933,0.996136,is_unbalance=True
2,Vanilla LGBM,2.239552,0.106899,2.346451,0.534490,2.774042,0.996071,is_unbalance=True
3,XGBoost (scale_pos_weight),4.904172,0.051910,4.956082,0.259547,5.163720,0.995964,scale_pos_weight=30.5102
4,Bagging 50% row subsample,2.689476,0.101059,2.790535,0.505293,3.194769,0.995671,is_unbalance=True


Higher binary accuracy indicates better predictive quality, while lower train/inference times indicate better efficiency. How does LightGBM compare with XGBoost in terms of accuracy in this case? How about speed?

What is the argument for why the GOSS sampling strategy should yield speed gains? How does this strategy go about maintaining accuracy? Does GOSS achieve the speed gains we would expect in this case? If no, why might that might be (hint: think about the steps required by that sampling algorithm)?

Explain in a couple sentences how the exclusive feature bundling algorithm decides what features to bundle. Why does bundling features accelerate model training?

It would be helpful for your cybersecurity operation to not just be able to identify whether or not there is an anomaly but also what *type* of attack is occurring. Luckily we can also use the LightGBM model on nonbinary classification problems.



In [8]:
label_encoder = LabelEncoder()
y_multi = label_encoder.fit_transform(y_multi_raw)

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X,
    y_multi,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_multi,
)

multi_clf = LGBMClassifier(
    objective="multiclass",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    n_jobs=-1,
)

t0 = time.perf_counter()
multi_clf.fit(X_train_multi, y_train_multi)
multi_train_seconds = time.perf_counter() - t0

t1 = time.perf_counter()
multi_preds = multi_clf.predict(X_test_multi)
multi_infer_seconds = time.perf_counter() - t1

multi_infer_full_est_seconds = multi_infer_seconds * (len(X) / len(X_test_multi))
multi_total_load_test_split = multi_train_seconds + multi_infer_seconds
multi_total_load_full_est = multi_train_seconds + multi_infer_full_est_seconds
multi_acc = accuracy_score(y_test_multi, multi_preds)

multi_metrics = {
    "Model": "Multiclass LGBM",
    "Train Time (s)": multi_train_seconds,
    "Inference Time (s)": multi_infer_seconds,
    "Total Load (s)": multi_total_load_test_split,
    "Estimated Full Inference Load (s)": multi_infer_full_est_seconds,
    "Estimated Full Total Load (s)": multi_total_load_full_est,
    "Multiclass Accuracy": multi_acc,
    "Imbalance Handling": "class_weight='balanced'",
    "Num Classes": len(label_encoder.classes_),
}

print("\nCompact class-wise report:")
print(classification_report(y_test_multi, multi_preds, target_names=label_encoder.classes_, zero_division=0))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017825 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5885
[LightGBM] [Info] Number of data points in the train set: 560000, number of used features: 79
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf



Compact class-wise report:
                precision    recall  f1-score   support

      Analysis       0.04      0.17      0.07       105
     Backdoors       0.04      0.11      0.06       107
           DoS       0.40      0.25      0.31       233
      Exploits       0.82      0.71      0.76      1082
       Fuzzers       0.64      0.80      0.71      1010
       Generic       0.99      0.96      0.98      1504
        Normal       1.00      1.00      1.00    135558
Reconnaissance       0.88      0.91      0.89       352
     Shellcode       0.74      0.87      0.80        45
         Worms       0.50      0.60      0.55         5

      accuracy                           0.99    140001
     macro avg       0.61      0.64      0.61    140001
  weighted avg       0.99      0.99      0.99    140001



## Export All Measurements to JSON

In [9]:
##TODO: Output training statistics to json
##If you've stored results in comparison_df and multi_metrics you shouldn't have to edit this cell.
output_payload = {
    "dataset": {
        "data_path": DATA_PATH,
        "feature_schema_path": FEATURE_PATH,
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
    },
    "binary_models": comparison_df.to_dict(orient="records"),
    "multiclass_model": multi_metrics,
}

json_output_path = "unsw_nb15_lightgbm_measurements.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=2)
